In [ ]:
### Plotting precipitation 

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
from pathlib import Path
import rasterio
import geopandas as gpd
from shapely.geometry import box

import requests
from PIL import Image
from io import BytesIO

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/ICON_Forecast/ICON-CH1-EPS_liestal_bl_2024_2024-06-25T15Z.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"
catchment_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/domain_Morges_alert_final.gpkg"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/MORGES_FORECAST_PLOTS/")
out_dir.mkdir(exist_ok=True, parents=True)

# plot extent in EPSG:2056
plot_extent = (2510227.993, 2540975.818, 1140564.666, 1172820.076)   # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Swisstopo WMS layer
wms_layer = "ch.swisstopo.swisstlm3d-karte-grau"

precip_alpha = 0.70
wms_start_resolution_m = 2
wms_max_pixels = 40_000_000
save_dpi = 400

# choose lead times
max_lead_hours = 6

# optionally force variable name
forced_var_name = None
# example:
# forced_var_name = "precipitation_amount"

# --------------------------------------------------
# Robust WMS fetch
# --------------------------------------------------
def get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=2,
    layer="ch.swisstopo.swisstlm3d-karte-grau",
    max_pixels=10_000_000
):
    xmin, xmax = float(min(xmin, xmax)), float(max(xmin, xmax))
    ymin, ymax = float(min(ymin, ymax)), float(max(ymin, ymax))
    dx, dy = xmax - xmin, ymax - ymin

    res = float(resolution_m)
    while True:
        width_px = int(np.ceil(dx / res))
        height_px = int(np.ceil(dy / res))
        if width_px * height_px <= max_pixels:
            break
        res *= 2

    bbox = f"{xmin},{ymin},{xmax},{ymax}"
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer,
        "BBOX": bbox,
        "CRS": "EPSG:2056",
        "WIDTH": width_px,
        "HEIGHT": height_px,
        "FORMAT": "image/png",
        "TRANSPARENT": "TRUE",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/png,image/*,*/*;q=0.8"
    }

    r = requests.get("https://wms.geo.admin.ch/", params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        print("Failed to fetch WMS:", r.status_code)
        return None, res

    ctype = r.headers.get("Content-Type", "")
    if "image" not in ctype.lower():
        print("WMS returned non-image content:", ctype)
        print(r.text[:250])
        return None, res

    return Image.open(BytesIO(r.content)).convert("RGBA"), res


# --------------------------------------------------
# LOAD FORECAST
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)

print(ds)
print("Data variables:", list(ds.data_vars))

if forced_var_name is not None:
    var_name = forced_var_name
else:
    var_name = list(ds.data_vars)[0]

da = ds[var_name]
print(f"Using variable: {var_name}")
print("Original dims:", da.dims)

# remove forecast_reference_time if present
if "forecast_reference_time" in da.dims:
    da = da.isel(forecast_reference_time=0)

# reorder
required_dims = {"lead_time", "realization", "y", "x"}
missing = required_dims - set(da.dims)
if missing:
    raise ValueError(f"Missing expected dimensions: {missing}")

da = da.transpose("lead_time", "realization", "y", "x")
print("Reordered dims:", da.dims)

# --------------------------------------------------
# SELECT FIRST 6 LEAD HOURS
# --------------------------------------------------
lead_time_hours = da["lead_time"] / np.timedelta64(1, "h")
lead_mask = (lead_time_hours > 0) & (lead_time_hours <= max_lead_hours)
da = da.sel(lead_time=da["lead_time"][lead_mask])

selected_hours = (da["lead_time"] / np.timedelta64(1, "h")).values
print("Selected lead times (h):", selected_hours)

# --------------------------------------------------
# READ DEM BOUNDS
# --------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# --------------------------------------------------
# LOAD CATCHMENTS
# --------------------------------------------------
catchment_gdf = gpd.read_file(catchment_file)

if catchment_gdf.crs is None:
    raise ValueError("Catchment file has no CRS.")

if catchment_gdf.crs.to_epsg() != 2056:
    catchment_gdf = catchment_gdf.to_crs("EPSG:2056")

bbox_geom = box(xmin, ymin, xmax, ymax)

try:
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:2056")
    catchment_plot = gpd.clip(catchment_gdf, bbox_gdf)
except Exception as e:
    print("gpd.clip failed, using intersects fallback.")
    print("Reason:", e)
    catchment_plot = catchment_gdf[catchment_gdf.intersects(bbox_geom)]

print("Catchment features loaded:", len(catchment_gdf))
print("Catchment features in plot window:", len(catchment_plot))

# --------------------------------------------------
# SPATIAL SUBSET
# --------------------------------------------------
x_ascending = bool(da.x.values[0] < da.x.values[-1])
y_ascending = bool(da.y.values[0] < da.y.values[-1])

x_slice = slice(xmin, xmax) if x_ascending else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if y_ascending else slice(ymax, ymin)

da_space = da.sel(x=x_slice, y=y_slice)

print("Subset shape after spatial selection:", da_space.shape)

if da_space.sizes["x"] == 0 or da_space.sizes["y"] == 0:
    raise ValueError("Spatial subset is empty. Check plot_extent vs forecast coordinates.")

# extent from selected coordinates
xvals = da_space.x.values
yvals = da_space.y.values
data_extent = (float(xvals.min()), float(xvals.max()), float(yvals.min()), float(yvals.max()))

# --------------------------------------------------
# COLOR LEVELS
# --------------------------------------------------
levels_full = np.array(
    [0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350],
    dtype=float
)

vmax = float(np.nanmax(da_space.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in selected forecast subset.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if len(levels) < 2:
    levels = np.array([0.1, max(1.0, vmax)])

if levels[-1] < vmax:
    higher_levels = levels_full[levels_full > levels[-1]]
    if len(higher_levels) > 0:
        levels = np.append(levels, higher_levels[0])
    else:
        levels = np.append(levels, vmax)

mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"Forecast vmax in window: {vmax:.2f}")
print("Levels used:", levels)

# --------------------------------------------------
# GET BACKGROUND ONCE
# --------------------------------------------------
bg_img, used_res = get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=wms_start_resolution_m,
    layer=wms_layer,
    max_pixels=wms_max_pixels
)
print(f"WMS resolution used: {used_res} m/px")

# --------------------------------------------------
# MORGES LOCATION
# --------------------------------------------------
morges_x = 2527132.921
morges_y = 1150707.111

# --------------------------------------------------
# PLOT PER ENSEMBLE AND LEAD TIME
# --------------------------------------------------
lead_hours = (da_space["lead_time"] / np.timedelta64(1, "h")).values
realizations = da_space["realization"].values

for ens in realizations:
    for lt, lt_h in zip(da_space["lead_time"].values, lead_hours):

        frame = da_space.sel(realization=ens, lead_time=lt).astype(float)
        frame = frame.where(frame > 0)

        fig, ax = plt.subplots(figsize=(8, 7))
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")

        # Background
        if bg_img is not None:
            ax.imshow(bg_img, extent=(xmin, xmax, ymin, ymax), origin="upper", zorder=0)

        # Forecast precipitation
        im = ax.imshow(
            frame.values,
            extent=data_extent,
            origin="lower",
            interpolation="nearest",
            cmap=cmap,
            norm=norm,
            alpha=precip_alpha,
            zorder=2
        )

        # Catchment outlines
        if len(catchment_plot) > 0:
            catchment_plot.boundary.plot(
                ax=ax,
                edgecolor="black",
                linewidth=1.2,
                zorder=4
            )

        # DEM rectangle
        rect = Rectangle(
            (dem_left, dem_bottom),
            dem_right - dem_left,
            dem_top - dem_bottom,
            fill=False,
            edgecolor="red",
            linewidth=2.0,
            zorder=5
        )
        ax.add_patch(rect)

        # Morges marker
        ax.scatter(
            morges_x,
            morges_y,
            marker="^",
            s=50,
            edgecolor="black",
            facecolor="yellow",
            linewidth=1.5,
            zorder=6
        )

        ax.text(
            morges_x + 500,
            morges_y + 500,
            "Morges",
            fontsize=8,
            color="black",
            weight="bold",
            zorder=6
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect("equal")

        title_txt = f"Ensemble {int(ens)} | Lead time +{float(lt_h):.0f} h"
        ax.set_title(title_txt, color="black")

        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
        cbar.set_label("precipitation (mm)")
        cbar.ax.tick_params(colors="black")
        cbar.outline.set_edgecolor("black")

        out = out_dir / f"ICON_MORGES_ens{int(ens):02d}_lead{int(round(float(lt_h))):02d}h.png"
        plt.savefig(out, dpi=save_dpi, transparent=True, bbox_inches="tight")
        plt.close()

print("Done. Saved to:", out_dir)

<xarray.Dataset> Size: 262MB
Dimensions:         (time: 16, y: 2527, x: 1617)
Coordinates:
  * time            (time) float32 64B 0.0 0.08333 0.1667 ... 1.083 1.167 1.25
  * y               (y) float32 10kB 1.197e+06 1.197e+06 ... 1.192e+06 1.192e+06
  * x               (x) float32 6kB 2.568e+06 2.568e+06 ... 2.571e+06 2.571e+06
Data variables:
    rainfall_depth  (time, y, x) float32 262MB ...
Attributes:
    description:  Spatially and temporally varying rainfall for TUFLOW model,...
    history:      Created on: 2025-01-25
    source:       Generated from Zell DEM and 30-year return period rainfall ...

In [1]:
import xarray as xr

path = "/storage/homefs/ge24z347/Liestal_event/Data_forprocess/COSMO/cosmo1e_2022-05-05T12.nc"

ds = xr.open_dataset(path, decode_cf=False)
ds

FileNotFoundError: [Errno 2] No such file or directory: '/storage/homefs/ge24z347/Liestal_event/Data_forprocess/COSMO/cosmo1e_2022-05-05T12.nc'

In [ ]:
ds.forecast_reference_time

<xarray.DataArray 'forecast_reference_time' (forecast_reference_time: 1)> Size: 8B
array([1.651752e+09])
Coordinates:
  * forecast_reference_time  (forecast_reference_time) float64 8B 1.652e+09
Attributes:
    _FillValue:  nan
    units:       seconds since 1970-01-01
    calendar:    proleptic_gregorian

In [ ]:
COSMO_flowdepth = xr.open_dataset("/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Zell_2m/Zell_2m_COSMO_2022-05-05T12-00.nc")
COSMO_flowdepth 

<xarray.Dataset> Size: 13GB
Dimensions:                  (forecast_reference_time: 1, lead_time: 11,
                              realization: 11, x: 2500, y: 1500)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2022...
  * lead_time                (lead_time) float64 88B 0.0 3.6e+03 ... 3.6e+04
  * realization              (realization) int32 44B 0 1 2 3 4 5 6 7 8 9 10
  * x                        (x) float64 20kB 2.702e+06 2.702e+06 ... 2.707e+06
  * y                        (y) float64 12kB 1.258e+06 1.258e+06 ... 1.255e+06
Data variables:
    time                     (forecast_reference_time, lead_time) datetime64[ns] 88B ...
    water_depth              (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    vel_x_c                  (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    vel_y_c                  (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    flux_x_c                 (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    flux_y_c                 (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    vel_mag                  (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
    flux_mag                 (forecast_reference_time, lead_time, realization, y, x) float32 2GB ...
Attributes:
    Conventions:  CF-1.8
    source:       LISFLOOD ASCII ensemble stacked to COSMO-like forecast form...
    crs:          EPSG:2056

In [ ]:
wd_r2 = COSMO_flowdepth["water_depth"].sel(realization=2)
wd_r2.values

array([[[[0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         ...,
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00]],

        [[0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         ...,
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.000e+00, 0.000e+00,
          0.000e+00],
         [0.000e+00, 0.000e+00, 0.000e+00, ..., 0.

In [ ]:

ds_flowdepth = xr.open_dataset("/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Zell_2m/Zell_2m_Combiprecip.nc")
ds_flowdepth  

<xarray.Dataset> Size: 1GB
Dimensions:         (REFERENCE_TS: 11, y: 1500, x: 2500)
Coordinates:
  * REFERENCE_TS    (REFERENCE_TS) datetime64[ns] 88B 2022-05-05T12:00:00 ......
  * y               (y) float64 12kB 1.258e+06 1.258e+06 ... 1.255e+06 1.255e+06
  * x               (x) float64 20kB 2.702e+06 2.702e+06 ... 2.707e+06 2.707e+06
    spatial_ref     int64 8B ...
Data variables:
    water_depth     (REFERENCE_TS, y, x) float32 165MB ...
    vel_x_c         (x, REFERENCE_TS, y) float32 165MB ...
    vel_y_c         (y, REFERENCE_TS, x) float32 165MB ...
    flux_x_c        (x, REFERENCE_TS, y) float32 165MB ...
    flux_y_c        (y, REFERENCE_TS, x) float32 165MB ...
    vel_mag         (x, REFERENCE_TS, y) float32 165MB ...
    flux_mag        (x, REFERENCE_TS, y) float32 165MB ...
    discharge_cell  (y, x, REFERENCE_TS) float32 165MB ...
    speed_cell      (y, x, REFERENCE_TS) float32 165MB ...
Attributes:
    Conventions:    CF-1.8
    source:         LISFLOOD single-member ASCII series, cell-centred on wate...
    deterministic:  true

In [ ]:
rain_6 = ds.rainfall_depth.isel(time=8)
rain_6.values

array([[4.1388073 , 4.1388073 , 4.1388073 , 4.1388073 , 4.1388073 ],
       [4.1388073 , 4.1388073 , 4.1388073 , 4.1388073 , 4.1388073 ],
       [3.54983807, 3.83302712, 3.83302712, 3.83302712, 4.1388073 ]])

In [2]:
import xarray as xr
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------
nc_file  = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/Combiprecip/CPC_00060_H_20220502000000_20220508230000.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Zell_2m/Zell_2m.dem"

out_dir  = Path("/storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation")
out_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Load data
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)
cpc = ds["CPC"]
time_coord = "REFERENCE_TS"

# --------------------------------------------------
# DEM bounds
# --------------------------------------------------
with rasterio.open(dem_file) as src:
    left, bottom, right, top = src.bounds

y_vals = cpc["y"]
x_vals = cpc["x"]

y_slice = slice(bottom, top) if y_vals[0] < y_vals[-1] else slice(top, bottom)
x_slice = slice(left, right) if x_vals[0] < x_vals[-1] else slice(right, left)

# --------------------------------------------------
# Times
# --------------------------------------------------
times = [
    "2022-05-05T12:00:00",
    "2022-05-05T13:00:00",
    "2022-05-05T14:00:00",
    "2022-05-05T15:00:00",
    "2022-05-05T16:00:00",
    "2022-05-05T17:00:00",
    "2022-05-05T18:00:00",
    "2022-05-05T19:00:00",
    "2022-05-05T20:00:00",
    "2022-05-05T21:00:00",
    "2022-05-05T22:00:00",
]

# --------------------------------------------------
# AUTO-DETECT MAXIMUM VALUE
# --------------------------------------------------
cpc_max = float(cpc.max())
vmax = int(np.ceil(cpc_max / 5) * 5)
print("Detected CPC max:", cpc_max, "---> Rounded to:", vmax)

# --------------------------------------------------
# BINS STARTING FROM 1
# --------------------------------------------------
# 1–5, 5–10, 10–15, ...
bounds = np.concatenate(([1], np.arange(5, vmax + 5, 5)))
n_bins = len(bounds)

# --------------------------------------------------
# COLORMAP
# --------------------------------------------------
mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

base_cmap = ListedColormap(mswiss_15)
continuous = plt.cm.get_cmap(base_cmap)
colors = continuous(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)

# --------------------------------------------------
# LOOP
# --------------------------------------------------
for t in times:
    cpc_t = cpc.sel({time_coord: t})
    cpc_win = cpc_t.sel(x=x_slice, y=y_slice)

    # Zero rainfall becomes transparent
    cpc_masked = cpc_win.where(cpc_win != 0)

    fig, ax = plt.subplots(figsize=(8, 6))

    im = cpc_masked.plot.pcolormesh(
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolors="k",
        linewidth=0.4,
        cbar_kwargs={
            "label": "precipitation amount (mm)",
            "ticks": bounds,   # Now: 1,5,10,15,...
        }
    )

    rect = Rectangle((left, bottom), right - left, top - bottom,
                     fill=False, edgecolor="red", linewidth=1.2)
    ax.add_patch(rect)

    ax.set_aspect("equal")
    ax.set_title(f"Precipitation at {t}")
    ax.set_xlabel("")
    ax.set_ylabel("")

    outfile = out_dir / f"CPC_{t.replace(':','')}.png"
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

    print("Saved:", outfile)

print("Done.")


Detected CPC max: 60.749366760253906 ---> Rounded to: 65


/tmp/ipykernel_2355463/1581960426.py:82: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  continuous = plt.cm.get_cmap(base_cmap)


Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T120000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T130000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T140000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T150000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T160000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T170000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T180000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combiprecip_Zell_2m_precipitation/CPC_2022-05-05T190000.png
Saved: /storage/homefs/ge24z347/Liestal_event/ZELL_PLOTS/Combipr

In [ ]:
#### MORGES PLOTTING FROM THE NETCDFILE 

In [15]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/Combiprecip/CPC_00060_H_20240624000000_20240630230000.nc"
var_name = "CPC"
time_coord = "REFERENCE_TS"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/MORGES_PLOTS/")
out_dir.mkdir(exist_ok=True)

# QGIS extent (EPSG:2056) from your screenshot:
plot_extent = (2510227.993, 2540975.818, 1140564.666, 1172820.076)  # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Exact times
times_exact = np.array([
    "2024-06-25T12:00:00",
    "2024-06-25T13:00:00",
    "2024-06-25T14:00:00",
    "2024-06-25T15:00:00",
    "2024-06-25T16:00:00",
    "2024-06-25T17:00:00",
    "2024-06-25T18:00:00",
    "2024-06-25T19:00:00",
    "2024-06-25T20:00:00",
    "2024-06-25T21:00:00",
    "2024-06-25T22:00:00",
], dtype="datetime64[ns]")

# --------------------------------------------------
# LOAD + SUBSET (space + exact times)
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)
da = ds[var_name]

x_slice = slice(xmin, xmax) if da.x[0] < da.x[-1] else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if da.y[0] < da.y[-1] else slice(ymax, ymin)

da_sel = da.sel(x=x_slice, y=y_slice).sel({time_coord: times_exact})

print("Selected times:")
print(da_sel[time_coord].values)

# --------------------------------------------------
# DISCRETE "RADAR-LIKE" LEVELS (good for small mm too)
# --------------------------------------------------
levels_full = np.array([0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350], dtype=float)
vmax = float(np.nanmax(da_sel.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in the selected extent/times.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if levels[-1] < vmax:
    levels = np.append(levels, levels_full[levels_full > levels[-1]][0])  # one level above vmax

# palette you used (expanded if needed)
mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]
n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))  # NaN transparent
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"vmax in window: {vmax:.2f} mm")
print("levels used:", levels)

# --------------------------------------------------
# PLOT (keep 1 km pixels)
# --------------------------------------------------
for t in da_sel[time_coord].values:
    frame = da_sel.sel({time_coord: t}).astype(float)
    frame = frame.where(frame > 0)  # 0 -> transparent

    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    im = ax.imshow(
        frame.values,
        extent=(xmin, xmax, ymin, ymax),
        origin="lower",
        interpolation="nearest",  # keep native 1km pixels
        cmap=cmap,
        norm=norm
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(str(np.datetime64(t))[:16], color="black")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
    cbar.set_label("precipitation (mm)")
    cbar.ax.tick_params(colors="black")
    cbar.outline.set_edgecolor("black")

    out = out_dir / f"CPC_{str(t)[:19].replace(':','')}.png"
    plt.savefig(out, dpi=250, transparent=True, bbox_inches="tight")
    plt.close()

print("Done. Saved to:", out_dir)

Selected times:
['2024-06-25T12:00:00.000000000' '2024-06-25T13:00:00.000000000'
 '2024-06-25T14:00:00.000000000' '2024-06-25T15:00:00.000000000'
 '2024-06-25T16:00:00.000000000' '2024-06-25T17:00:00.000000000'
 '2024-06-25T18:00:00.000000000' '2024-06-25T19:00:00.000000000'
 '2024-06-25T20:00:00.000000000' '2024-06-25T21:00:00.000000000'
 '2024-06-25T22:00:00.000000000']
vmax in window: 60.75 mm
levels used: [ 0.1  0.2  0.5  1.   2.   3.   5.   7.  10.  15.  20.  30.  40.  50.
 60.  80. ]
Done. Saved to: /storage/homefs/ge24z347/Zell_event/MORGES_PLOTS


In [21]:
#### Morges precipitation with background 

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path

import requests
from PIL import Image
from io import BytesIO

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/Combiprecip/CPC_00060_H_20240624000000_20240630230000.nc"
var_name = "CPC"
time_coord = "REFERENCE_TS"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/MORGES_PLOTS/")
out_dir.mkdir(exist_ok=True)

# QGIS extent (EPSG:2056)
plot_extent = (2510227.993, 2540975.818, 1140564.666, 1172820.076)  # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Exact times
times_exact = np.array([
    "2024-06-25T12:00:00",
    "2024-06-25T13:00:00",
    "2024-06-25T14:00:00",
    "2024-06-25T15:00:00",
    "2024-06-25T16:00:00",
    "2024-06-25T17:00:00",
    "2024-06-25T18:00:00",
    "2024-06-25T19:00:00",
    "2024-06-25T20:00:00",
    "2024-06-25T21:00:00",
    "2024-06-25T22:00:00",
], dtype="datetime64[ns]")

# Swisstopo WMS layer
wms_layer = "ch.swisstopo.swisstlm3d-karte-grau"

precip_alpha = 0.70

wms_start_resolution_m = 2     # try 1 or 2 for sharper
wms_max_pixels = 40_000_000    # was 10_000_000
save_dpi = 400                # was 250

# --------------------------------------------------
# Robust WMS fetch (auto-coarsen if image would be huge)
# --------------------------------------------------
def get_swisstopo_background_image(xmin, xmax, ymin, ymax,
                                   resolution_m=2,
                                   layer="ch.swisstopo.swisstlm3d-karte-grau",
                                   max_pixels=10_000_000):
    xmin, xmax = float(min(xmin, xmax)), float(max(xmin, xmax))
    ymin, ymax = float(min(ymin, ymax)), float(max(ymin, ymax))
    dx, dy = xmax - xmin, ymax - ymin

    res = float(resolution_m)
    while True:
        width_px = int(np.ceil(dx / res))
        height_px = int(np.ceil(dy / res))
        if width_px * height_px <= max_pixels:
            break
        res *= 2  # coarsen until safe

    bbox = f"{xmin},{ymin},{xmax},{ymax}"
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer,
        "BBOX": bbox,
        "CRS": "EPSG:2056",
        "WIDTH": width_px,
        "HEIGHT": height_px,
        "FORMAT": "image/png",
        "TRANSPARENT": "TRUE",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/png,image/*,*/*;q=0.8"
    }

    r = requests.get("https://wms.geo.admin.ch/", params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        print(" Failed to fetch WMS:", r.status_code)
        return None, res

    # sometimes WMS returns XML error with HTTP 200
    ctype = r.headers.get("Content-Type", "")
    if "image" not in ctype.lower():
        print(" WMS returned non-image content:", ctype)
        print(r.text[:250])
        return None, res

    return Image.open(BytesIO(r.content)).convert("RGBA"), res


# --------------------------------------------------
# LOAD + SUBSET (space + exact times)
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)
da = ds[var_name]

x_slice = slice(xmin, xmax) if da.x[0] < da.x[-1] else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if da.y[0] < da.y[-1] else slice(ymax, ymin)

da_sel = da.sel(x=x_slice, y=y_slice).sel({time_coord: times_exact})

print("Selected times:")
print(da_sel[time_coord].values)

# --------------------------------------------------
# DISCRETE "RADAR-LIKE" LEVELS (good for small mm too)
# --------------------------------------------------
levels_full = np.array([0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350], dtype=float)
vmax = float(np.nanmax(da_sel.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in the selected extent/times.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if levels[-1] < vmax:
    levels = np.append(levels, levels_full[levels_full > levels[-1]][0])  # one level above vmax

mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))  # NaN transparent
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"vmax in window: {vmax:.2f} mm")
print("levels used:", levels)

# --------------------------------------------------
# Get background ONCE
# --------------------------------------------------
bg_img, used_res = get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=wms_start_resolution_m,
    layer=wms_layer,
    max_pixels=wms_max_pixels
)
print(f"WMS resolution used: {used_res} m/px")

# --------------------------------------------------
# PLOT (keep 1 km pixels) + WMS behind + alpha=0.3
# --------------------------------------------------
for t in da_sel[time_coord].values:
    frame = da_sel.sel({time_coord: t}).astype(float)
    frame = frame.where(frame > 0)  # 0 -> transparent

    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    # background
    if bg_img is not None:
        ax.imshow(bg_img, extent=(xmin, xmax, ymin, ymax), origin="upper", zorder=0)
    else:
        print(" No WMS background for", t)

    # precipitation overlay (alpha=0.3)
    im = ax.imshow(
        frame.values,
        extent=(xmin, xmax, ymin, ymax),
        origin="lower",
        interpolation="nearest",  # keep native 1km pixels
        cmap=cmap,
        norm=norm,
        alpha=precip_alpha,
        zorder=2
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(str(np.datetime64(t))[:16], color="black")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
    cbar.set_label("precipitation (mm)")
    cbar.ax.tick_params(colors="black")
    cbar.outline.set_edgecolor("black")
    
    
    # --------------------------------------------------
    # ADD MORGES LOCATION
    # --------------------------------------------------

    morges_x = 2527132.921
    morges_y = 1150707.111

    # Campground marker (triangle)
    ax.scatter(
        morges_x,
        morges_y,
        marker="^",          # triangle
        s=120,               # size (increase if needed)
        edgecolor="black",
        facecolor="yellow",
        linewidth=1.5,
        zorder=6
        )

    # label
    ax.text(
        morges_x + 500,     # small offset so text doesn't overlap point
        morges_y + 500,
        "Morges",
        fontsize=8,
        color="black",
        weight="bold",
        zorder=6
        )

    out = out_dir / f"CPC_BG_{str(t)[:19].replace(':','')}.png"
    plt.savefig(out, dpi=save_dpi, transparent=True, bbox_inches="tight")
    plt.close()

print("Done. Saved to:", out_dir)

Selected times:
['2024-06-25T12:00:00.000000000' '2024-06-25T13:00:00.000000000'
 '2024-06-25T14:00:00.000000000' '2024-06-25T15:00:00.000000000'
 '2024-06-25T16:00:00.000000000' '2024-06-25T17:00:00.000000000'
 '2024-06-25T18:00:00.000000000' '2024-06-25T19:00:00.000000000'
 '2024-06-25T20:00:00.000000000' '2024-06-25T21:00:00.000000000'
 '2024-06-25T22:00:00.000000000']
vmax in window: 60.75 mm
levels used: [ 0.1  0.2  0.5  1.   2.   3.   5.   7.  10.  15.  20.  30.  40.  50.
 60.  80. ]
WMS resolution used: 8.0 m/px
Done. Saved to: /storage/homefs/ge24z347/Zell_event/MORGES_PLOTS


In [ ]:
###MORGES code 

In [3]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
from pathlib import Path
import rasterio
import geopandas as gpd
from shapely.geometry import box

import requests
from PIL import Image
from io import BytesIO

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
nc_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/Combiprecip/CPC_00060_H_20240624000000_20240630230000.nc"
dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Morges_2m_CP/Morges_2m_CP.dem"
catchment_file = "/storage/homefs/ge24z347/Zell_event/Data_forprocess/geo_ezgg_40km.gpkg"

var_name = "CPC"
time_coord = "REFERENCE_TS"

out_dir = Path("/storage/homefs/ge24z347/Zell_event/MORGES_PLOTS/")
out_dir.mkdir(exist_ok=True)

# QGIS extent (EPSG:2056)
plot_extent = (2510227.993, 2540975.818, 1140564.666, 1172820.076)   # xmin, xmax, ymin, ymax
xmin, xmax, ymin, ymax = plot_extent

# Exact times
times_exact = np.array([
    "2024-06-25T12:00:00",
    "2024-06-25T13:00:00",
    "2024-06-25T14:00:00",
    "2024-06-25T15:00:00",
    "2024-06-25T16:00:00",
    "2024-06-25T17:00:00",
    "2024-06-25T18:00:00",
    "2024-06-25T19:00:00",
    "2024-06-25T20:00:00",
    "2024-06-25T21:00:00",
    "2024-06-25T22:00:00",
], dtype="datetime64[ns]")

# Swisstopo WMS layer
wms_layer = "ch.swisstopo.swisstlm3d-karte-grau"

precip_alpha = 0.70
wms_start_resolution_m = 2
wms_max_pixels = 40_000_000
save_dpi = 400

# --------------------------------------------------
# Robust WMS fetch (auto-coarsen if image would be huge)
# --------------------------------------------------
def get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=2,
    layer="ch.swisstopo.swisstlm3d-karte-grau",
    max_pixels=10_000_000
):
    xmin, xmax = float(min(xmin, xmax)), float(max(xmin, xmax))
    ymin, ymax = float(min(ymin, ymax)), float(max(ymin, ymax))
    dx, dy = xmax - xmin, ymax - ymin

    res = float(resolution_m)
    while True:
        width_px = int(np.ceil(dx / res))
        height_px = int(np.ceil(dy / res))
        if width_px * height_px <= max_pixels:
            break
        res *= 2

    bbox = f"{xmin},{ymin},{xmax},{ymax}"
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer,
        "BBOX": bbox,
        "CRS": "EPSG:2056",
        "WIDTH": width_px,
        "HEIGHT": height_px,
        "FORMAT": "image/png",
        "TRANSPARENT": "TRUE",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/png,image/*,*/*;q=0.8"
    }

    r = requests.get("https://wms.geo.admin.ch/", params=params, headers=headers, timeout=60)
    if r.status_code != 200:
        print("Failed to fetch WMS:", r.status_code)
        return None, res

    ctype = r.headers.get("Content-Type", "")
    if "image" not in ctype.lower():
        print("WMS returned non-image content:", ctype)
        print(r.text[:250])
        return None, res

    return Image.open(BytesIO(r.content)).convert("RGBA"), res


# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
ds = xr.open_dataset(nc_file, decode_times=True)

print("Coordinate uniqueness:")
print(f"  time unique: {ds.indexes[time_coord].is_unique}")
print(f"  x unique:    {ds.indexes['x'].is_unique}")
print(f"  y unique:    {ds.indexes['y'].is_unique}")

# Remove duplicated time values if needed
if not ds.indexes[time_coord].is_unique:
    _, unique_idx = np.unique(ds[time_coord].values, return_index=True)
    ds = ds.isel({time_coord: np.sort(unique_idx)})
    print(f"Removed duplicate time entries. New time length: {ds.sizes[time_coord]}")

da = ds[var_name]

# --------------------------------------------------
# READ DEM BOUNDS
# --------------------------------------------------
with rasterio.open(dem_file) as src:
    dem_left, dem_bottom, dem_right, dem_top = src.bounds

print("DEM bounds:")
print(f"left={dem_left}, right={dem_right}, bottom={dem_bottom}, top={dem_top}")

# --------------------------------------------------
# LOAD CATCHMENT
# --------------------------------------------------
catchment_gdf = gpd.read_file(catchment_file)

if catchment_gdf.crs is None:
    raise ValueError("Catchment file has no CRS. Please define it first.")

if catchment_gdf.crs.to_string() != "EPSG:2056":
    catchment_gdf = catchment_gdf.to_crs("EPSG:2056")

# optional clip to plotting window for faster drawing
bbox_geom = box(xmin, ymin, xmax, ymax)

# First try gpd.clip, and if that fails use intersects fallback
try:
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:2056")
    catchment_plot = gpd.clip(catchment_gdf, bbox_gdf)
except Exception as e:
    print("gpd.clip failed, using intersects fallback.")
    print("Reason:", e)
    catchment_plot = catchment_gdf[catchment_gdf.intersects(bbox_geom)]

print("Catchment features loaded:", len(catchment_gdf))
print("Catchment features in plot window:", len(catchment_plot))

# --------------------------------------------------
# SPATIAL SUBSET
# --------------------------------------------------
x_ascending = bool(da.x.values[0] < da.x.values[-1])
y_ascending = bool(da.y.values[0] < da.y.values[-1])

x_slice = slice(xmin, xmax) if x_ascending else slice(xmax, xmin)
y_slice = slice(ymin, ymax) if y_ascending else slice(ymax, ymin)

da_space = da.sel(x=x_slice, y=y_slice)

print("Subset shape after spatial selection:", da_space.shape)

if da_space.sizes["x"] == 0 or da_space.sizes["y"] == 0:
    raise ValueError("Spatial subset is empty. Check plot_extent vs NetCDF coordinates.")

# extent from actual selected raster coordinates
xvals = da_space.x.values
yvals = da_space.y.values
data_extent = (float(xvals.min()), float(xvals.max()), float(yvals.min()), float(yvals.max()))

# --------------------------------------------------
# TIME SUBSET (safe selection)
# --------------------------------------------------
available_times = da_space[time_coord].values
common_times = np.intersect1d(available_times, times_exact)

if len(common_times) == 0:
    raise ValueError("None of the requested times were found in the dataset.")

missing_times = np.setdiff1d(times_exact, available_times)
if len(missing_times) > 0:
    print("These requested times were NOT found and will be skipped:")
    print(missing_times)

da_sel = da_space.sel({time_coord: common_times})

print("Selected times:")
print(da_sel[time_coord].values)

# --------------------------------------------------
# DISCRETE "RADAR-LIKE" LEVELS
# --------------------------------------------------
levels_full = np.array(
    [0.1, 0.2, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 60, 80, 100, 130, 160, 250, 350],
    dtype=float
)

vmax = float(np.nanmax(da_sel.values))
if not np.isfinite(vmax) or vmax <= 0:
    raise ValueError("No positive precipitation values in the selected extent/times.")

levels = levels_full[levels_full <= max(1.0, vmax)]
if len(levels) < 2:
    levels = np.array([0.1, max(1.0, vmax)])

if levels[-1] < vmax:
    higher_levels = levels_full[levels_full > levels[-1]]
    if len(higher_levels) > 0:
        levels = np.append(levels, higher_levels[0])
    else:
        levels = np.append(levels, vmax)

mswiss_15 = [
    "#F9844A", "#FBB476", "#FDD8A3",
    "#FFFFCC", "#F0E68C",
    "#C7E9B4", "#7FCDBB",
    "#41B6C4", "#1D91C0",
    "#225EA8", "#253494",
    "#081D58", "#31183b",
    "#70265c", "#bc3754"
]

n_bins = len(levels) - 1
base = plt.get_cmap(ListedColormap(mswiss_15))
colors = base(np.linspace(0, 1, n_bins))
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

print(f"vmax in window: {vmax:.2f} mm")
print("levels used:", levels)

# --------------------------------------------------
# GET BACKGROUND ONCE
# --------------------------------------------------
bg_img, used_res = get_swisstopo_background_image(
    xmin, xmax, ymin, ymax,
    resolution_m=wms_start_resolution_m,
    layer=wms_layer,
    max_pixels=wms_max_pixels
)
print(f"WMS resolution used: {used_res} m/px")

# --------------------------------------------------
# MORGES LOCATION
# --------------------------------------------------
morges_x = 2527132.921
morges_y = 1150707.111

# --------------------------------------------------
# PLOT
# --------------------------------------------------
for t in da_sel[time_coord].values:
    frame = da_sel.sel({time_coord: t}).astype(float)
    frame = frame.where(frame > 0)

    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_alpha(0)
    ax.set_facecolor("none")

    # Background
    if bg_img is not None:
        ax.imshow(bg_img, extent=(xmin, xmax, ymin, ymax), origin="upper", zorder=0)
    else:
        print("No WMS background for", t)

    # Precipitation
    im = ax.imshow(
        frame.values,
        extent=data_extent,
        origin="lower",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        alpha=precip_alpha,
        zorder=2
    )

    # Catchment outline
    if len(catchment_plot) > 0:
        catchment_plot.boundary.plot(
            ax=ax,
            edgecolor="black",
            linewidth=1.2,
            zorder=4
        )

    # DEM rectangle
    rect = Rectangle(
        (dem_left, dem_bottom),
        dem_right - dem_left,
        dem_top - dem_bottom,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        zorder=5
    )
    ax.add_patch(rect)

    # Morges marker
    ax.scatter(
        morges_x,
        morges_y,
        marker="^",
        s=50,
        edgecolor="black",
        facecolor="yellow",
        linewidth=1.5,
        zorder=6
    )

    ax.text(
        morges_x + 500,
        morges_y + 500,
        "Morges",
        fontsize=8,
        color="black",
        weight="bold",
        zorder=6
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(str(np.datetime64(t))[:16], color="black")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=levels)
    cbar.set_label("precipitation (mm)")
    cbar.ax.tick_params(colors="black")
    cbar.outline.set_edgecolor("black")

    out = out_dir / f"CPC_MORGES_{str(t)[:19].replace(':','')}.png"
    plt.savefig(out, dpi=save_dpi, transparent=True, bbox_inches="tight")
    plt.close()

print("Done. Saved to:", out_dir)

Coordinate uniqueness:
  time unique: True
  x unique:    True
  y unique:    True
DEM bounds:
left=2518000.0, right=2529000.0, bottom=1150000.0, top=1161000.0
Catchment features loaded: 1193
Catchment features in plot window: 30
Subset shape after spatial selection: (168, 32, 31)
Selected times:
['2024-06-25T12:00:00.000000000' '2024-06-25T13:00:00.000000000'
 '2024-06-25T14:00:00.000000000' '2024-06-25T15:00:00.000000000'
 '2024-06-25T16:00:00.000000000' '2024-06-25T17:00:00.000000000'
 '2024-06-25T18:00:00.000000000' '2024-06-25T19:00:00.000000000'
 '2024-06-25T20:00:00.000000000' '2024-06-25T21:00:00.000000000'
 '2024-06-25T22:00:00.000000000']
vmax in window: 60.75 mm
levels used: [ 0.1  0.2  0.5  1.   2.   3.   5.   7.  10.  15.  20.  30.  40.  50.
 60.  80. ]
WMS resolution used: 8.0 m/px
Done. Saved to: /storage/homefs/ge24z347/Zell_event/MORGES_PLOTS
